In [ ]:
import os
os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # Q-Agent: max_new_tokens=1024, plus system+user prompt
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "unsloth/gpt-oss-20b-BF16"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
#### Unsloth: `hf_xet==1.1.10` and `ipykernel>6.30.1` breaks progress bars. Disabling for now in XET.
#### Unsloth: To re-enable progress bars, please downgrade to `ipykernel==6.30.1` or wait for a fix to
https://github.com/huggingface/xet-core/issues/526
INFO 02-14 17:44:45 [__init__.py:225] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.9: Fast Gpt_Oss patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Tri

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Model loaded: unsloth/gpt-oss-20b-BF16
Parameter count: 20,914,757,184


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                        # LoRA rank (higher = more capacity)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,               # 2x rank for faster convergence
    lora_dropout=0,              # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Making `model.base_model.model.model` require gradients
Trainable: 15,925,248 / 20,930,682,432 (0.08%)


In [4]:
import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/8d27e9782cd1a1d5645b8a639eb84b8ed7f868de"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/qagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/qagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

Train: 113,451 samples
Val:   12,605 samples
Sample keys: ['messages']

Sample messages[0]:
  [system]: You are a competitive question generator for logical reasoning challenges. 
Generate a challenging m...
  [user]: Generate a challenging blood relations MCQ involving 5 relationship hops. The chain should be long a...
  [assistant]: {"topic": "Blood Relations and Family Tree/Family tree logic", "question": "Read the following infor...


In [5]:
# Convert to HuggingFace Datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

Train dataset: Dataset({
    features: ['messages'],
    num_rows: 113451
})
Val dataset:   Dataset({
    features: ['messages'],
    num_rows: 12605
})


In [6]:
batch = val_dataset[0]
batch.keys()

dict_keys(['messages'])

In [7]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize message format (handles role name normalization)
train_dataset = standardize_sharegpt(train_dataset)
val_dataset = standardize_sharegpt(val_dataset)

# Formatting function: apply model's native chat template
def formatting_prompts_func(examples):
    # standardize_sharegpt may rename 'messages' to 'conversations'
    key = "conversations" if "conversations" in examples else "messages"
    convos = examples[key]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Preview formatted text
print("═" * 60)
print("Sample formatted training text:")
print("═" * 60)
print(train_dataset["text"][0][:500])
print("...")

Map:   0%|          | 0/113451 [00:00<?, ? examples/s]

Map:   0%|          | 0/12605 [00:00<?, ? examples/s]

════════════════════════════════════════════════════════════
Sample formatted training text:
════════════════════════════════════════════════════════════
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-14

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a competitive question generator for logical reasoning challenges. 
Generate a challenging multiple-choice question (MCQ) in 
...


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq

# ═══ Training Configuration ═══
OUTPUT_DIR = "qagent_gptoss_outputs"
NUM_EPOCHS = 1        # Start with 1, increase if val loss still dropping
BATCH_SIZE = 128       # MI300X can handle more; tune based on seq_length
GRAD_ACCUM = 8        # Effective batch = 128 * 8 = 1024
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,  # Don't pack — preserve conversation boundaries
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # For ROCm stability
        remove_unused_columns=True,
    ),
)

# Train ONLY on assistant responses — mask system + user tokens from loss
# NOTE: gpt-oss uses <|channel|>final between "assistant" and "<|message|>"
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

print(f"Training config:")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total train samples: {len(train_dataset):,}")
print(f"  Total val samples:   {len(val_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Steps/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/113451 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/12605 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/113451 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/12605 [00:00<?, ? examples/s]

Training config:
  Effective batch size: 1024
  Total train samples: 113,451
  Total val samples:   12,605
  Epochs: 1
  Steps/epoch: ~110


In [9]:
gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_mem} GB")
print(f"Reserved before training: {reserved} GB")
print(f"Available for KV cache + training: {total_mem - reserved:.1f} GB")

GPU: 
Total VRAM: 255.69 GB
Reserved before training: 39.46 GB
Available for KV cache + training: 216.2 GB


In [ ]:
# Switch to training mode
FastLanguageModel.for_training(model)

# Train
trainer_stats = trainer.train()

# Print final stats
print(f"\nTraining completed!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"  Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
peak_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"  Peak VRAM: {peak_mem} GB / {total_mem} GB ({100*peak_mem/total_mem:.1f}%)")

In [ ]:
# Save LoRA adapters (small, ~100-300MB)
LORA_PATH = "qagent_gptoss_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapters saved to: {LORA_PATH}")

# Save merged 16-bit model (full size, for vLLM deployment)
MERGED_PATH = "qagent_gptoss_merged_16bit"
print(f"Saving merged 16-bit model to: {MERGED_PATH} ...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {MERGED_PATH}")

In [ ]:
# ═══ Q-Agent Test Inference — one per category ═══
import json

FastLanguageModel.for_inference(model)

system_prompt = (
    "You are a competitive question generator for logical reasoning and aptitude tests.\n"
    "Generate a single multiple-choice question (MCQ) with exactly 4 options (A–D).\n"
    'Return ONLY a JSON object with keys: "topic", "question", "choices" (dict A-D), "answer" (letter), "explanation".'
)

# One test prompt per category with specific question_type
test_prompts = [
    # Syllogisms — both_neither_conclusion
    (
        "Generate a syllogism MCQ where the student must decide which conclusions follow "
        "from the given statements, including 'Both follow' and 'Neither follows' as options.\n"
        "Topic: Logical Reasoning/Syllogisms\n"
        "Question Type: both_neither_conclusion"
    ),
    # Blood Relations — complex_relation_4hop
    (
        "Generate a blood relation MCQ requiring the solver to navigate 4 relationship hops "
        "(e.g., A→B→C→D→E) to determine the final relationship.\n"
        "Topic: Logical Reasoning/Blood Relations\n"
        "Question Type: complex_relation_4hop"
    ),
    # Seating Arrangement — circular_position_query
    (
        "Generate a circular seating arrangement MCQ where the student must determine "
        "who sits at a specific position relative to another person.\n"
        "Topic: Logical Reasoning/Seating Arrangements\n"
        "Question Type: circular_position_query"
    ),
    # Series — numeric_next_term
    (
        "Generate a number series MCQ where the student must find the next term.\n"
        "Topic: Logical Reasoning/Series\n"
        "Question Type: numeric_next_term"
    ),
]

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

    print(f"\n{'═'*60}")
    print(f"PROMPT {i+1}: {prompt.split(chr(10))[-1]}")
    print(f"{'─'*60}")
    print(response[:600])

    # Quick structural validation
    try:
        parsed = json.loads(response)
        required = {"topic", "question", "choices", "answer", "explanation"}
        missing = required - set(parsed.keys())
        if missing:
            print(f"⚠ Missing keys: {missing}")
        elif len(parsed.get("choices", {})) != 4:
            print(f"⚠ Expected 4 choices, got {len(parsed.get('choices', {}))}")
        elif parsed.get("answer") not in ("A", "B", "C", "D"):
            print(f"⚠ Invalid answer letter: {parsed.get('answer')}")
        else:
            print(f"✓ Valid JSON — topic: {parsed['topic']}, answer: {parsed['answer']}")
    except json.JSONDecodeError:
        print(f"✗ Failed to parse as JSON")

In [ ]:
import json
from collections import defaultdict

# ═══ Full Validation — Structural Quality Evaluation (BATCHED) ═══
with open(f"{DATA_DIR}/qagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating Q-Agent on FULL validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 16  # Lower than A-Agent since max_new_tokens=1024
REQUIRED_KEYS = {"topic", "question", "choices", "answer", "explanation"}
VALID_ANSWERS = {"A", "B", "C", "D"}

# ── Step 1: Pre-process all samples ──
samples = []
for item in val_raw:
    msgs = item["messages"]
    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    if "syllogism" in user_content.lower():
        topic_key = "Syllogisms"
    elif "blood" in user_content.lower():
        topic_key = "Blood Relations"
    elif "seating" in user_content.lower():
        topic_key = "Seating Arrangements"
    elif "series" in user_content.lower():
        topic_key = "Series"
    else:
        topic_key = "Other"
    samples.append({"topic_key": topic_key, "prompt_msgs": msgs[:-1]})

print(f"Prepared {len(samples)} samples for batched inference")

# ── Step 2: Left-pad tokenizer for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Step 3: Batched inference ──
total = 0
json_ok = 0
struct_ok = 0
parse_fails = 0
key_fails = 0
choice_fails = 0
answer_fails = 0
topic_stats = defaultdict(lambda: {"total": 0, "json_ok": 0, "struct_ok": 0})

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in range(num_batches):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    # Tokenize all prompts in one go
    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Decode and validate each response
    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        total += 1
        topic_stats[s["topic_key"]]["total"] += 1

        # 1. JSON parseable?
        try:
            parsed = json.loads(response)
        except json.JSONDecodeError:
            parse_fails += 1
            continue

        json_ok += 1
        topic_stats[s["topic_key"]]["json_ok"] += 1

        # 2. Has all required keys?
        if not REQUIRED_KEYS.issubset(parsed.keys()):
            key_fails += 1
            continue

        # 3. Exactly 4 choices?
        choices = parsed.get("choices", {})
        if not isinstance(choices, dict) or len(choices) != 4:
            choice_fails += 1
            continue

        # 4. Valid answer letter in A-D?
        if parsed.get("answer") not in VALID_ANSWERS:
            answer_fails += 1
            continue

        # All checks passed
        struct_ok += 1
        topic_stats[s["topic_key"]]["struct_ok"] += 1

    # Progress
    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        pct = 100 * struct_ok / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] struct_ok: {struct_ok}/{total} ({pct:.1f}%)  json_fails: {parse_fails}")

# Restore padding side
tokenizer.padding_side = orig_padding_side

# ═══ Final Results ═══
print(f"\n{'═'*60}")
print(f"Q-AGENT VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"JSON parse success:     {json_ok}/{total} = {100*json_ok/total:.1f}%")
print(f"Structurally valid:     {struct_ok}/{total} = {100*struct_ok/total:.1f}%")
print(f"\nFailure breakdown:")
print(f"  JSON parse failures:  {parse_fails}")
print(f"  Missing keys:         {key_fails}")
print(f"  Wrong # choices:      {choice_fails}")
print(f"  Invalid answer letter:{answer_fails}")
print(f"\nPer-Topic Breakdown (structurally valid %):")
for topic, stats in sorted(topic_stats.items()):
    pct = 100 * stats["struct_ok"] / stats["total"] if stats["total"] > 0 else 0
    jpct = 100 * stats["json_ok"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['struct_ok']:5d}/{stats['total']:5d} = {pct:.1f}%  (json_ok: {jpct:.1f}%)")